In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Implementation of a Multi-Layer Perceptron (MLP) for Multi-Class Image Classification using PyTorch

## 1. Setup and Imports

In [33]:
!pip install torch torchvision torchaudio matplotlib seaborn scikit-learn
!pip install "scikit-learn<1.4"

In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import os
import time


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


### Create Plot Directory and Helper Function

In [35]:
plot_dir = '/content/drive/MyDrive/MLP_Plots'
os.makedirs(plot_dir, exist_ok=True)
print(f"Plot directory created at: {plot_dir}")

def save_plot(fig, filename, dpi=600):
    filepath = os.path.join(plot_dir, filename)
    fig.savefig(filepath, dpi=dpi, bbox_inches='tight', format='pdf')
    plt.close(fig)
    print(f"Plot saved to {filepath}")

Plot directory created at: /content/drive/MyDrive/MLP_Plots


## 2. Task 1: Dataset Exploration

### Load Fashion-MNIST Dataset and Print Dimensions

In [36]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Testing dataset size: {len(test_dataset)}")
print(f"Training images shape: {train_dataset.data.shape}")
print(f"Training labels shape: {train_dataset.targets.shape}")
print(f"Testing images shape: {test_dataset.data.shape}")
print(f"Testing labels shape: {test_dataset.targets.shape}")

class_names = train_dataset.classes
print(f"Class names: {class_names}")

Training dataset size: 60000
Testing dataset size: 10000
Training images shape: torch.Size([60000, 28, 28])
Training labels shape: torch.Size([60000])
Testing images shape: torch.Size([10000, 28, 28])
Testing labels shape: torch.Size([10000])
Class names: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


### Display Ten Sample Fashion-MNIST Images

In [37]:
fig = plt.figure(figsize=(10, 10))
for i in range(10):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_dataset.data[i], cmap=plt.cm.binary)
    plt.xlabel(class_names[train_dataset.targets[i].item()])
plt.suptitle('10 Sample Fashion-MNIST Images', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
save_plot(fig, 'Sample_Fashion_MNIST_Images.pdf')
plt.show()

print("Inference: This plot displays a selection of 10 images from the Fashion-MNIST dataset, showing the variety of apparel items and their corresponding class labels. This helps in understanding the visual characteristics of the dataset.")

Plot saved to /content/drive/MyDrive/MLP_Plots/Sample_Fashion_MNIST_Images.pdf
Inference: This plot displays a selection of 10 images from the Fashion-MNIST dataset, showing the variety of apparel items and their corresponding class labels. This helps in understanding the visual characteristics of the dataset.


### Plot Class Distribution

In [38]:
class_counts = torch.bincount(train_dataset.targets)


fig = plt.figure(figsize=(10, 6))
plt.bar(class_names, class_counts.cpu().numpy(), color='skyblue')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.title('Fashion-MNIST Training Data Class Distribution')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
save_plot(fig, 'Fashion_MNIST_Class_Distribution.pdf')
plt.show()

print("Class distribution in training data:")
for i, count in enumerate(class_counts):
    print(f"{class_names[i]}: {count.item()} images")

print("\nInference: This bar plot illustrates that the Fashion-MNIST training dataset has a balanced distribution across all 10 classes, with approximately 6,000 images per class. This equal representation is beneficial for training unbiased classification models.")

Plot saved to /content/drive/MyDrive/MLP_Plots/Fashion_MNIST_Class_Distribution.pdf
Class distribution in training data:
T-shirt/top: 6000 images
Trouser: 6000 images
Pullover: 6000 images
Dress: 6000 images
Coat: 6000 images
Sandal: 6000 images
Shirt: 6000 images
Sneaker: 6000 images
Bag: 6000 images
Ankle boot: 6000 images

Inference: This bar plot illustrates that the Fashion-MNIST training dataset has a balanced distribution across all 10 classes, with approximately 6,000 images per class. This equal representation is beneficial for training unbiased classification models.


## 3. Task 2: Data Preprocessing

### Flatten Images, Normalize Pixels, and One-Hot Encode Labels

In [39]:
train_images = train_dataset.data
train_labels = train_dataset.targets
test_images = test_dataset.data
test_labels = test_dataset.targets

print("Shapes before preprocessing:")
print(f"Training images shape: {train_images.shape}")
print(f"Training labels shape: {train_labels.shape}")
print(f"Testing images shape: {test_images.shape}")
print(f"Testing labels shape: {test_labels.shape}")

train_images_flat = train_images.view(train_images.shape[0], -1)
test_images_flat = test_images.view(test_images.shape[0], -1)

train_images_normalized = train_images_flat.float() / 255.0
test_images_normalized = test_images_flat.float() / 255.0

num_classes = len(class_names)
train_labels_one_hot = torch.nn.functional.one_hot(train_labels, num_classes=num_classes).float()
test_labels_one_hot = torch.nn.functional.one_hot(test_labels, num_classes=num_classes).float()

print("\nShapes after preprocessing:")
print(f"Flattened and normalized training images shape: {train_images_normalized.shape}")
print(f"Flattened and normalized testing images shape: {test_images_normalized.shape}")
print(f"One-hot encoded training labels shape: {train_labels_one_hot.shape}")
print(f"One-hot encoded testing labels shape: {test_labels_one_hot.shape}")

train_dataset_processed = TensorDataset(train_images_normalized, train_labels)
test_dataset_processed = TensorDataset(test_images_normalized, test_labels)

Shapes before preprocessing:
Training images shape: torch.Size([60000, 28, 28])
Training labels shape: torch.Size([60000])
Testing images shape: torch.Size([10000, 28, 28])
Testing labels shape: torch.Size([10000])

Shapes after preprocessing:
Flattened and normalized training images shape: torch.Size([60000, 784])
Flattened and normalized testing images shape: torch.Size([10000, 784])
One-hot encoded training labels shape: torch.Size([60000, 10])
One-hot encoded testing labels shape: torch.Size([10000, 10])


## 4. Task 3: Model Construction

### Construct Baseline MLP Model

In [40]:
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, num_classes):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size2, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        return out


input_size = 28 * 28
hidden_size1 = 128
hidden_size2 = 64
num_classes = len(class_names)


model = MLP(input_size, hidden_size1, hidden_size2, num_classes).to(device)


print(model)

MLP(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)


## 5. Task 4: Model Training

### Compile and Train the Model

In [41]:
learning_rate = 0.001
batch_size = 32
epochs = 20

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

train_loader = DataLoader(dataset=train_dataset_processed, batch_size=batch_size, shuffle=True)

train_size = int(0.8 * len(train_dataset_processed))
val_size = len(train_dataset_processed) - train_size
train_subset, val_subset = random_split(train_dataset_processed, [train_size, val_size])

train_loader_split = DataLoader(dataset=train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_subset, batch_size=batch_size, shuffle=False)

history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}

print(f"Training model for {epochs} epochs with batch size {batch_size} and learning rate {learning_rate}")

model.train()
start_time = time.time()

for epoch in range(epochs):
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for i, (images, labels) in enumerate(train_loader_split):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    train_loss_avg = running_loss / len(train_loader_split)
    val_loss_avg = val_loss / len(val_loader)
    train_accuracy = correct_train / total_train
    val_accuracy = correct_val / total_val

    history['loss'].append(train_loss_avg)
    history['val_loss'].append(val_loss_avg)
    history['accuracy'].append(train_accuracy)
    history['val_accuracy'].append(val_accuracy)

    print (f'Epoch [{epoch+1}/{epochs}], Loss: {train_loss_avg:.4f}, Accuracy: {train_accuracy:.4f}, Val Loss: {val_loss_avg:.4f}, Val Accuracy: {val_accuracy:.4f}')

end_time = time.time()
training_time = end_time - start_time
print(f"\nTraining complete in {training_time:.2f} seconds")

Training model for 20 epochs with batch size 32 and learning rate 0.001
Epoch [1/20], Loss: 0.5534, Accuracy: 0.8034, Val Loss: 0.4115, Val Accuracy: 0.8522
Epoch [2/20], Loss: 0.3924, Accuracy: 0.8583, Val Loss: 0.3928, Val Accuracy: 0.8608
Epoch [3/20], Loss: 0.3522, Accuracy: 0.8703, Val Loss: 0.3395, Val Accuracy: 0.8778
Epoch [4/20], Loss: 0.3272, Accuracy: 0.8797, Val Loss: 0.3165, Val Accuracy: 0.8862
Epoch [5/20], Loss: 0.3072, Accuracy: 0.8858, Val Loss: 0.3319, Val Accuracy: 0.8753
Epoch [6/20], Loss: 0.2930, Accuracy: 0.8899, Val Loss: 0.3059, Val Accuracy: 0.8899
Epoch [7/20], Loss: 0.2803, Accuracy: 0.8957, Val Loss: 0.3111, Val Accuracy: 0.8892
Epoch [8/20], Loss: 0.2677, Accuracy: 0.8988, Val Loss: 0.3245, Val Accuracy: 0.8832
Epoch [9/20], Loss: 0.2571, Accuracy: 0.9034, Val Loss: 0.2986, Val Accuracy: 0.8958
Epoch [10/20], Loss: 0.2473, Accuracy: 0.9068, Val Loss: 0.3047, Val Accuracy: 0.8964
Epoch [11/20], Loss: 0.2403, Accuracy: 0.9080, Val Loss: 0.3010, Val Accuracy

## 6. Task 5: Model Evaluation

### Compute Metrics and Display Confusion Matrix

In [42]:
model.eval()

test_loader = DataLoader(dataset=test_dataset_processed, batch_size=batch_size, shuffle=False)

all_labels = []
all_predictions = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

true_labels = np.array(all_labels)
predicted_labels = np.array(all_predictions)

accuracy = accuracy_score(true_labels, predicted_labels)
precision = precision_score(true_labels, predicted_labels, average='weighted', zero_division=0)
recall = recall_score(true_labels, predicted_labels, average='weighted', zero_division=0)
f1 = f1_score(true_labels, predicted_labels, average='weighted', zero_division=0)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1-score: {f1:.4f}")

print("\nClassification Report:\n")
print(classification_report(true_labels, predicted_labels, target_names=class_names, zero_division=0))

cm = confusion_matrix(true_labels, predicted_labels)

fig = plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Baseline Model')
save_plot(fig, 'Baseline_Confusion_Matrix.pdf')
plt.show()

print("\nInference: This confusion matrix visually represents the performance of the baseline model. Diagonal elements indicate correctly classified instances, while off-diagonal elements show misclassifications. It helps identify which classes are frequently confused with each other.")

Test Accuracy: 0.8786
Test Precision: 0.8818
Test Recall: 0.8786
Test F1-score: 0.8796

Classification Report:

              precision    recall  f1-score   support

 T-shirt/top       0.82      0.83      0.82      1000
     Trouser       0.97      0.98      0.98      1000
    Pullover       0.75      0.83      0.79      1000
       Dress       0.93      0.84      0.88      1000
        Coat       0.81      0.78      0.79      1000
      Sandal       0.98      0.95      0.96      1000
       Shirt       0.68      0.72      0.70      1000
     Sneaker       0.93      0.97      0.95      1000
         Bag       0.99      0.94      0.97      1000
  Ankle boot       0.97      0.95      0.96      1000

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000

Plot saved to /content/drive/MyDrive/MLP_Plots/Baseline_Confusion_Matrix.pdf

Inference: This confusion matrix visually represent

## 7. Hyperparameter Optimization

In [43]:
!pip install skorch
import skorch
from skorch import NeuralNetClassifier

### Flexible MLP Model for Hyperparameter Tuning

In [44]:
import torch.nn.functional as F

class FlexibleMLP(nn.Module):
    def __init__(self, input_size=784, num_classes=10,
                 hidden_layers=1, hidden_neurons=128,
                 activation='relu', dropout_rate=0.0):
        super().__init__()
        self.input_size = input_size
        self.num_classes = num_classes
        self.hidden_layers = hidden_layers
        self.hidden_neurons = hidden_neurons
        self.dropout_rate = dropout_rate

        self.layers = nn.ModuleList()

        self.layers.append(nn.Linear(input_size, hidden_neurons))

        for _ in range(hidden_layers - 1):
            self.layers.append(nn.Linear(hidden_neurons, hidden_neurons))

        self.layers.append(nn.Linear(hidden_neurons, num_classes))

        self.activation_fn = self._get_activation_fn(activation)
        self.dropout = nn.Dropout(dropout_rate)

    def _get_activation_fn(self, activation_name):
        if activation_name == 'relu':
            return nn.ReLU()
        elif activation_name == 'tanh':
            return nn.Tanh()
        elif activation_name == 'sigmoid':
            return nn.Sigmoid()
        else:
            raise ValueError(f"Unknown activation function: {activation_name}")

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            if i < len(self.layers) - 1:
                x = layer(x)
                x = self.activation_fn(x)
                if self.dropout_rate > 0:
                    x = self.dropout(x)
            else:
                x = layer(x)
        return x

print("FlexibleMLP class defined for hyperparameter tuning.")

FlexibleMLP class defined for hyperparameter tuning.


### Define the Hyperparameter Search Space

In [50]:
net = NeuralNetClassifier(
    FlexibleMLP,
    criterion=nn.CrossEntropyLoss,
    optimizer=optim.Adam,
    iterator_train__shuffle=True,
    max_epochs=10,
    batch_size=32,
    device=device,
    verbose=0
)


param_grid = {
    'module__hidden_layers': [1, 2, 3],
    'module__hidden_neurons': [32, 64, 128, 256],
    'lr': [0.1, 0.01, 0.001],
    'batch_size': [16, 32, 64, 128],
    'max_epochs': [5, 10],
    'optimizer': [optim.SGD, optim.Adam, optim.RMSprop],
    'module__activation': ['relu', 'tanh', 'sigmoid'],
    'module__dropout_rate': [0.0, 0.2, 0.5],
}

print("Hyperparameter search space defined:")
for param, values in param_grid.items():
    print(f"- {param}: {values}")

Hyperparameter search space defined:
- module__hidden_layers: [1, 2, 3]
- module__hidden_neurons: [32, 64, 128, 256]
- lr: [0.1, 0.01, 0.001]
- batch_size: [16, 32, 64, 128]
- max_epochs: [5, 10]
- optimizer: [<class 'torch.optim.sgd.SGD'>, <class 'torch.optim.adam.Adam'>, <class 'torch.optim.rmsprop.RMSprop'>]
- module__activation: ['relu', 'tanh', 'sigmoid']
- module__dropout_rate: [0.0, 0.2, 0.5]


### Perform Randomized Search with 5-fold Cross-Validation

In [51]:
from sklearn.model_selection import RandomizedSearchCV

X_train_hp = train_images_normalized.cpu().numpy()
y_train_hp = train_labels.cpu().numpy()


random_search = RandomizedSearchCV(
    estimator=net,
    param_distributions=param_grid,
    n_iter=2,
    cv=5,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

print("Starting Randomized Search... (This may take a while)")
search_start_time = time.time()
random_search.fit(X_train_hp, y_train_hp)
search_end_time = time.time()

print("Randomized Search completed.")

best_params = random_search.best_params_
best_score = random_search.best_score_

print(f"\nBest hyperparameters found: {best_params}")
print(f"Best cross-validation accuracy: {best_score:.4f}")
print(f"Randomized search took {search_end_time - search_start_time:.2f} seconds.")

Starting Randomized Search... (This may take a while)
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Randomized Search completed.

Best hyperparameters found: {'optimizer': <class 'torch.optim.adam.Adam'>, 'module__hidden_neurons': 128, 'module__hidden_layers': 1, 'module__dropout_rate': 0.2, 'module__activation': 'tanh', 'max_epochs': 10, 'lr': 0.001, 'batch_size': 128}
Best cross-validation accuracy: 0.8760
Randomized search took 174.78 seconds.


### 8.1. Train Optimized Model

In [ ]:
best_lr = best_params['lr']
best_batch_size = best_params['batch_size']
best_max_epochs = best_params['max_epochs']
best_optimizer = best_params['optimizer']
best_hidden_layers = best_params['module__hidden_layers']
best_hidden_neurons = best_params['module__hidden_neurons']
best_activation = best_params['module__activation']
best_dropout_rate = best_params['module__dropout_rate']

print("Training optimized model with the following parameters:")
print(f"  Learning Rate: {best_lr}")
print(f"  Batch Size: {best_batch_size}")
print(f"  Max Epochs: {best_max_epochs}")
print(f"  Optimizer: {best_optimizer.__name__}")
print(f"  Hidden Layers: {best_hidden_layers}")
print(f"  Hidden Neurons: {best_hidden_neurons}")
print(f"  Activation: {best_activation}")
print(f"  Dropout Rate: {best_dropout_rate}")

optimized_mlp_module = FlexibleMLP(
    input_size=input_size,
    num_classes=num_classes,
    hidden_layers=best_hidden_layers,
    hidden_neurons=best_hidden_neurons,
    activation=best_activation,
    dropout_rate=best_dropout_rate
)

optimized_net = NeuralNetClassifier(
    optimized_mlp_module,
    criterion=nn.CrossEntropyLoss,
    optimizer=best_optimizer,
    lr=best_lr,
    max_epochs=best_max_epochs,
    batch_size=best_batch_size,
    iterator_train__shuffle=True,
    device=device,
    verbose=0
)

X_train_opt = train_images_normalized.cpu().numpy()
y_train_opt = train_labels.cpu().numpy()

X_test_opt = test_images_normalized.cpu().numpy()
y_test_opt = test_labels.cpu().numpy()

print("\nStarting training of optimized model...")
optimized_train_start_time = time.time()
optimized_net.fit(X_train_opt, y_train_opt)
optimized_train_end_time = time.time()
optimized_training_time = optimized_train_end_time - optimized_train_start_time
print(f"Optimized model training completed in {optimized_training_time:.2f} seconds.")

### 8.2. Evaluate Optimized Model

In [ ]:
optimized_y_pred = optimized_net.predict(X_test_opt)

optimized_accuracy = accuracy_score(y_test_opt, optimized_y_pred)
optimized_precision = precision_score(y_test_opt, optimized_y_pred, average='weighted', zero_division=0)
optimized_recall = recall_score(y_test_opt, optimized_y_pred, average='weighted', zero_division=0)
optimized_f1 = f1_score(y_test_opt, optimized_y_pred, average='weighted', zero_division=0)

print(f"\nOptimized Model Test Accuracy: {optimized_accuracy:.4f}")
print(f"Optimized Model Test Precision: {optimized_precision:.4f}")
print(f"Optimized Model Test Recall: {optimized_recall:.4f}")
print(f"Optimized Model Test F1-score: {optimized_f1:.4f}")

print("\nOptimized Model Classification Report:\n")
print(classification_report(y_test_opt, optimized_y_pred, target_names=class_names, zero_division=0))

optimized_cm = confusion_matrix(y_test_opt, optimized_y_pred)

fig_opt_cm = plt.figure(figsize=(10, 8))
sns.heatmap(optimized_cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Optimized Model')
save_plot(fig_opt_cm, 'Optimized_Confusion_Matrix.pdf')
plt.show()

print("Inference: This confusion matrix for the optimized model shows the distribution of correct and incorrect predictions, indicating if hyperparameter tuning improved classification performance across different classes compared to the baseline.")

### 8.3. Plots for Optimized Model Training History

In [ ]:
optimized_history = optimized_net.history

# Get keys from the first record in history
if optimized_history:
    print("Optimized model history keys:")
    for key in optimized_history[0].keys():
        print(f"- {key}")

# Now that 'metrics=['acc']' has been added, 'train_acc' and 'valid_acc' should be present.
# Assuming 'train_loss', 'valid_loss', 'train_acc', 'valid_acc' are the correct keys.
epochs_optimized = range(1, len(optimized_history[:, 'train_loss']) + 1)

fig_acc = plt.figure(figsize=(10, 6))
plt.plot(epochs_optimized, optimized_history[:, 'train_acc'], label='Training Accuracy')
plt.plot(epochs_optimized, optimized_history[:, 'valid_acc'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Optimized Model Training & Validation Accuracy vs. Epoch')
plt.legend()
plt.grid(True)
save_plot(fig_acc, 'Optimized_Model_Accuracy_vs_Epoch.pdf')
plt.show()

print("Inference: This plot displays how the optimized model's training and validation accuracy evolved over epochs. It helps to identify if the model is overfitting or underfitting, and if the hyperparameter tuning led to better generalization.")

fig_loss = plt.figure(figsize=(10, 6))
plt.plot(epochs_optimized, optimized_history[:, 'train_loss'], label='Training Loss')
plt.plot(epochs_optimized, optimized_history[:, 'valid_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Optimized Model Training & Validation Loss vs. Epoch')
plt.legend()
plt.grid(True)
save_plot(fig_loss, 'Optimized_Model_Loss_vs_Epoch.pdf')
plt.show()

print("Inference: This plot shows the optimized model's training and validation loss over epochs. A decreasing trend in both indicates learning, while a divergence (validation loss increasing) suggests overfitting.")

### 8.4. Hyperparameter Search Results Plot

In [ ]:
results = pd.DataFrame(random_search.cv_results_)

results = results.sort_values(by='mean_test_score', ascending=False)

fig_hp, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.scatterplot(x='param_lr', y='mean_test_score', hue='param_optimizer', data=results, s=200, ax=axes[0], palette='viridis')
axes[0].set_xscale('log')
axes[0].set_title('Mean Test Score vs. Learning Rate (by Optimizer)')
axes[0].set_xlabel('Learning Rate (log scale)')
axes[0].set_ylabel('Mean Test Score')
axes[0].grid(True, which='both', ls='--')

sns.scatterplot(x='param_module__hidden_neurons', y='mean_test_score', hue='param_module__hidden_layers', data=results, s=200, ax=axes[1], palette='magma')
axes[1].set_title('Mean Test Score vs. Hidden Neurons (by Hidden Layers)')
axes[1].set_xlabel('Number of Hidden Neurons')
axes[1].set_ylabel('Mean Test Score')
axes[1].grid(True, which='both', ls='--')

plt.suptitle('Hyperparameter Search Results (Mean Cross-Validation Accuracy)', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
save_plot(fig_hp, 'Hyperparameter_Search_Results.pdf')
plt.show()

print("Inference: These plots visualize the relationship between different hyperparameters (e.g., learning rate, hidden neurons) and the mean cross-validation test score. They help in understanding which parameter ranges contribute to better model performance and can guide future optimization efforts.")

### 8.5. Best Model Accuracy Comparison Plot

In [ ]:
comparison_accuracies = pd.DataFrame({
    'Model': ['Baseline', 'Optimized'],
    'Accuracy': [accuracy, optimized_accuracy]
})

fig_comp = plt.figure(figsize=(8, 5))
sns.barplot(x='Model', y='Accuracy', data=comparison_accuracies, palette=['skyblue', 'lightcoral'])
plt.ylim(0.8, 1.0)
plt.title('Baseline vs. Optimized Model Test Accuracy')
plt.ylabel('Test Accuracy')
plt.grid(axis='y', linestyle='--', alpha=0.7)
save_plot(fig_comp, 'Model_Accuracy_Comparison.pdf')
plt.show()

print("Inference: This bar chart directly compares the test accuracy of the baseline model against the optimized model. It provides a clear visual indication of whether the hyperparameter optimization process successfully improved the model's performance on unseen data.")

## 9. Results

### Best Hyperparameters

| Parameter                 | Value |
| :------------------------ | :---- |
| Hidden Layers             |

In [ ]:
print(f"best_hidden_layers" + str(best_hidden_layers) + " |")
print(f"| Hidden Neurons            | " + str(best_hidden_neurons) + " |")
print(f"| Learning Rate             | " + str(best_lr) + " |")
print(f"| Batch Size                | " + str(best_batch_size) + " |")
print(f"| Optimizer                 | " + str(best_optimizer.__name__) + " |")
print(f"| Activation Function       | " + str(best_activation) + " |")
print(f"| Epochs                    | " + str(best_max_epochs) + " |")
print(f"| Dropout                   | " + str(best_dropout_rate) + " |")
print(f"| Cross-validation Accuracy | " + f"{best_score:.4f}" + " |")
print(f"| Testing Accuracy          | " + f"{optimized_accuracy:.4f}" + " |")

### Performance Comparison

| Metric        | Baseline | Optimized |
| :------------ | :------- | :-------- |
| Accuracy      |

In [ ]:
print(f"{accuracy:.4f}   | " + f"{optimized_accuracy:.4f} |")
print(f"| Precision     | " + f"{precision:.4f}   | " + f"{optimized_precision:.4f} |")
print(f"| Recall        | " + f"{recall:.4f}   | " + f"{optimized_recall:.4f} |")
print(f"| F1-score      | " + f"{f1:.4f}   | " + f"{optimized_f1:.4f} |")
print(f"| Training Time | " + f"{training_time:.2f}s | " + f"{optimized_training_time:.2f}s |")